# KNN Sınıflandırması — Wine Veri Seti

Bu notebook'ta K-En Yakın Komşu (KNN) algoritmasını Wine veri seti üzerinde uyguluyoruz.

**Hedef:** Kimyasal analizlere göre şarabın hangi üreticiden geldiğini tahmin etmek (3 sınıf)

## 1. Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA

## 2. Veri Seti

In [ ]:
wine = load_wine()

df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['class'] = wine.target
df['class_name'] = df['class'].map({0: 'Üretici 1', 1: 'Üretici 2', 2: 'Üretici 3'})

print("Veri seti boyutu:", df.shape)
print("\nÖzellikler:", list(wine.feature_names))
print("\nSınıf dağılımı:")
print(df['class_name'].value_counts())
df.head()

In [ ]:
df.describe().round(2)

## 3. Keşifsel Veri Analizi (EDA)

In [ ]:
# Önemli özelliklerin kutu grafiği
key_features = ['alcohol', 'malic_acid', 'flavanoids', 'color_intensity', 'proline']
colors = ['#e74c3c', '#3498db', '#2ecc71']
labels = ['Üretici 1', 'Üretici 2', 'Üretici 3']

fig, axes = plt.subplots(1, len(key_features), figsize=(18, 5))

for ax, feature in zip(axes, key_features):
    data_to_plot = [df[df['class'] == i][feature].values for i in range(3)]
    bp = ax.boxplot(data_to_plot, patch_artist=True, labels=labels)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(feature, fontsize=10)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Temel Özelliklerin Üreticilere Göre Dağılımı', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wine_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Korelasyon ısı haritası
plt.figure(figsize=(12, 9))
sns.heatmap(df[wine.feature_names].corr(), annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.4, annot_kws={'size': 8})
plt.title('Özellikler Arası Korelasyon Matrisi', fontsize=13)
plt.tight_layout()
plt.savefig('wine_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Ön İşleme

In [ ]:
X = df[wine.feature_names].values
y = df['class'].values

# Min-Max Normalizasyon
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# Train-Test Split (%70 eğitim, %30 test)
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Eğitim seti : {X_train.shape[0]} örnek")
print(f"Test seti   : {X_test.shape[0]} örnek")
print(f"Özellik sayısı: {X_train.shape[1]}")

## 5. En İyi k Değerini Bulma

In [ ]:
k_values = range(1, 21)
train_scores = []
test_scores = []
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    test_scores.append(knn.score(X_test, y_test))
    cv = cross_val_score(knn, X_normalized, y, cv=5)
    cv_scores.append(cv.mean())

best_k = list(k_values)[np.argmax(cv_scores)]
print(f"En iyi k değeri (5-fold CV): {best_k}")
print(f"CV doğruluğu               : {max(cv_scores):.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(k_values, train_scores, 'o-', label='Eğitim', color='#3498db')
plt.plot(k_values, test_scores, 's-', label='Test', color='#e74c3c')
plt.plot(k_values, cv_scores, '^--', label='5-Fold CV', color='#2ecc71')
plt.axvline(x=best_k, color='gray', linestyle='--', alpha=0.7, label=f'En iyi k={best_k}')
plt.xlabel('k (Komşu Sayısı)')
plt.ylabel('Doğruluk')
plt.title('k Değerine Göre KNN Performansı — Wine')
plt.legend()
plt.xticks(k_values)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('wine_k_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Model Eğitimi ve Değerlendirme

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test Doğruluğu: {acc:.4f} ({acc*100:.2f}%)")
print(f"Doğru tahmin  : {np.sum(y_pred == y_test)} / {len(y_test)}")

In [ ]:
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred, target_names=labels))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, linewidths=0.5)
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title(f'Confusion Matrix — Wine (k={best_k})', fontsize=13)
plt.tight_layout()
plt.savefig('wine_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. PCA ile 2D Görselleştirme

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_normalized)

plt.figure(figsize=(8, 6))
colors_map = {0: '#e74c3c', 1: '#3498db', 2: '#2ecc71'}
for cls, label in zip([0, 1, 2], labels):
    mask = y == cls
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors_map[cls], label=label, alpha=0.7, edgecolors='white', s=60)

plt.xlabel(f'PCA Bileşen 1 (Varyans: {pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PCA Bileşen 2 (Varyans: {pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Wine Veri Seti — PCA ile 2D Görselleştirme')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('wine_pca.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Açıklanan toplam varyans: {sum(pca.explained_variance_ratio_)*100:.1f}%")

## 8. Sonuç

| Metrik | Değer |
|---|---|
| Algoritma | K-En Yakın Komşu (KNN) |
| Veri Seti | Wine (178 örnek, 13 özellik, 3 sınıf) |
| En iyi k | CV ile belirlendi |
| Normalizasyon | Min-Max Scaler |
| Test Doğruluğu | Notebook çalıştırıldığında görünür |